# Task 2 — Credit Card Fraud Detection
**CodSoft Machine Learning Internship**

Goal: classify credit card transactions as fraudulent or legitimate.

**Before running:** download the dataset from the link in the CodSoft task
PDF and place the CSV in this same folder. This notebook auto-detects which
of the two common versions of this dataset you have:
- The classic anonymized dataset (`Time, V1..V28, Amount, Class`)
- The richer transactions dataset (`merchant, category, amt, ..., is_fraud`)

If your file isn't named `creditcard.csv`, just change `DATA_PATH` below.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    ConfusionMatrixDisplay,
)
import joblib

DATA_PATH = "creditcard.csv"

## 1. Load the data + detect which schema we have

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

if "Class" in df.columns:
    SCHEMA = "classic"
    target_col = "Class"
elif "is_fraud" in df.columns:
    SCHEMA = "rich"
    target_col = "is_fraud"
else:
    raise ValueError(
        "Couldn't find a 'Class' or 'is_fraud' target column — open the CSV "
        "and check the actual label column name, then set target_col manually."
    )

print(f"\nDetected schema: {SCHEMA} | target column: {target_col}")
df.head()

## 2. Class balance
Fraud datasets are almost always heavily imbalanced — this matters a lot for
which metrics are meaningful (accuracy alone is misleading here).

In [ ]:
counts = df[target_col].value_counts()
print(counts)
print(f"\nFraud rate: {counts.get(1, 0) / len(df) * 100:.3f}%")

plt.figure(figsize=(5, 4))
counts.plot(kind="bar", color=["steelblue", "indianred"])
plt.title("Legitimate (0) vs Fraudulent (1) transactions")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("fraud_class_balance.png", dpi=100)
plt.show()

## 3. Feature preparation
Branches depending on which schema was detected.

In [ ]:
if SCHEMA == "classic":
    # Numeric PCA features (V1-V28) are already scaled; Amount/Time are not.
    features = df.drop(columns=[target_col])
    scaler = StandardScaler()
    for col in ["Amount", "Time"]:
        if col in features.columns:
            features[col] = scaler.fit_transform(features[[col]])
    X = features
    y = df[target_col]

else:  # "rich" schema
    drop_cols = [
        c
        for c in [
            "trans_date_trans_time", "cc_num", "merchant", "first", "last",
            "street", "job", "dob", "trans_num", "unix_time", target_col,
        ]
        if c in df.columns
    ]
    features = df.drop(columns=drop_cols)
    # Encode categorical columns
    for col in features.select_dtypes(include="object").columns:
        features[col] = LabelEncoder().fit_transform(features[col].astype(str))
    scaler = StandardScaler()
    numeric_cols = features.select_dtypes(include=np.number).columns
    features[numeric_cols] = scaler.fit_transform(features[numeric_cols])
    X = features
    y = df[target_col]

print("Final feature matrix shape:", X.shape)

## 4. Train / test split
`stratify=y` is important given the imbalance — otherwise a random split can
end up with almost no fraud cases in one of the sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 5. Train models
`class_weight="balanced"` tells the model to pay more attention to the rare
fraud class instead of just predicting "legitimate" every time.

In [ ]:
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)
log_preds = log_reg.predict(X_test)

In [ ]:
rf = RandomForestClassifier(
    n_estimators=150, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)

## 6. Evaluate
**Why not just look at accuracy?** If only 0.2% of transactions are fraud, a
model that predicts "legitimate" for everything scores ~99.8% accuracy while
catching zero fraud. Precision/recall/F1 and ROC-AUC tell the real story.

In [ ]:
print("=== Logistic Regression ===")
print(classification_report(y_test, log_preds, digits=3))
print("ROC-AUC:", roc_auc_score(y_test, log_reg.predict_proba(X_test)[:, 1]))

print("\n=== Random Forest ===")
print(classification_report(y_test, rf_preds, digits=3))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, log_preds, ax=ax[0], colorbar=False)
ax[0].set_title("Logistic Regression")
ConfusionMatrixDisplay.from_predictions(y_test, rf_preds, ax=ax[1], colorbar=False)
ax[1].set_title("Random Forest")
plt.tight_layout()
plt.savefig("fraud_confusion_matrices.png", dpi=100)
plt.show()

## 7. Feature importance (Random Forest)
Nice chart to show in your demo — which features drive the fraud decision.

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(8, 5))
importances.plot(kind="barh")
plt.title("Top 10 most important features")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("fraud_feature_importance.png", dpi=100)
plt.show()

## 8. Save the best model

In [ ]:
best_model = rf if roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1]) >= roc_auc_score(y_test, log_reg.predict_proba(X_test)[:, 1]) else log_reg
joblib.dump(best_model, "fraud_detection_model.joblib")
print("Saved fraud_detection_model.joblib")

## Conclusion
Random Forest with balanced class weights typically catches meaningfully
more fraud than plain Logistic Regression at the cost of some extra false
positives — a trade-off worth mentioning in your writeup. Next steps you
could note: try SMOTE oversampling, or tune the classification threshold
instead of using the default 0.5 cutoff.